# 06 — Tracing with LangSmith

## Why this notebook exists

In **notebook 05** we built an LLM-as-judge that can score open-ended agent outputs against a rubric. We can now *measure* whether an agent is doing well. But when a score drops, we can't yet *see* why: which step produced bad output? Where did the agent spend its time? Which span consumed the most tokens?

Traditional logging is too coarse — a wall of text with no structure. What we need is **tracing**: a hierarchical record of every function call in a run, with its inputs, outputs, latency, and token usage captured per span. LangSmith provides exactly that, for free, through a hosted UI that requires no infrastructure from us.

This notebook instruments a small multi-step agent with LangSmith tracing, shows you how to read the resulting span tree, demonstrates how a trace localises a bug to a single failing span, and surfaces per-run latency and token counts. At the end, the agent outputs we've been scoring in blind eval land will have a full call graph attached.

## What you'll learn

- How to set up LangSmith tracing with three env vars (`OPENAI_API_KEY`, `LANGCHAIN_API_KEY`, `LANGCHAIN_TRACING_V2=true`) and a free hosted account.
- How the `@traceable` decorator from the `langsmith` SDK turns any Python function into a traced span — inputs, outputs, latency, and token counts captured automatically.
- How LangChain's `ChatOpenAI` auto-traces without any decorator when `LANGCHAIN_TRACING_V2=true` is set.
- How to build a small multi-step agent where each step is its own span, giving you a **nested span tree** in the LangSmith UI: parent run → child spans → per-span details.
- How to use a trace to **debug a broken run**: the UI (and the run object) pinpoint which span raised an error or produced malformed output, so you don't have to re-read logs.
- Where LangSmith surfaces **latency, prompt tokens, completion tokens, and cost** for each run, and how to read these programmatically via the `Client`.
- The names of two open-source alternatives (Langfuse, Arize Phoenix) so you know they exist.

## 1. Setup + Env-Var Guard

### Prerequisites

This notebook requires:

| Variable | Purpose |
|---|---|
| `OPENAI_API_KEY` | Calls to `gpt-4o-mini` for the traced agent |
| `LANGCHAIN_API_KEY` | Your LangSmith API key |
| `LANGCHAIN_TRACING_V2` | Must be the string `"true"` to enable tracing |
| `LANGCHAIN_PROJECT` *(optional)* | Groups traces under a named project in the UI; defaults to `"default"` |

**LangSmith account (free tier):**
1. Go to **https://smith.langchain.com** and sign up for a free account.
2. Navigate to **Settings → API Keys** and create a key.
3. Copy the key and set it as `LANGCHAIN_API_KEY` in your shell or `.env` file.

The guard cell below checks all required variables and tells you exactly what is missing before any LLM call is made.

> **Gotcha:** As of mid-2025 the LangSmith SDK checks both `LANGSMITH_TRACING_V2` and `LANGCHAIN_TRACING_V2` (in that priority order). Either name works; `LANGCHAIN_TRACING_V2` is the historically common name in LangChain-based guides. Similarly, `LANGCHAIN_API_KEY` and `LANGSMITH_API_KEY` are both accepted. Always check the current [LangSmith quickstart docs](https://docs.smith.langchain.com/setup) if traces are not appearing — the SDK's `get_env_var()` utility resolves both namespaces automatically.

Install dependencies if needed (run once per environment):

```bash
pip install langsmith langchain langchain-openai openai python-dotenv
```

In [ ]:
import os

from dotenv import load_dotenv
load_dotenv()  # loads OPENAI_API_KEY + LANGCHAIN_API_KEY from project .env if present

# ---------------------------------------------------------------------------
# ENV-VAR GUARD — all required variables must be present before we proceed.
# The LangSmith SDK checks LANGSMITH_* and LANGCHAIN_* namespaces; we guard
# the LANGCHAIN_* names here because they are the historically common names.
# ---------------------------------------------------------------------------
REQUIRED = {
    "OPENAI_API_KEY": (
        "Your OpenAI API key. Get one at https://platform.openai.com/api-keys"
    ),
    "LANGCHAIN_API_KEY": (
        "Your LangSmith API key. Sign up free at https://smith.langchain.com, "
        "then go to Settings → API Keys."
    ),
    "LANGCHAIN_TRACING_V2": (
        'Must be set to the string "true" to enable LangSmith tracing. '
        "Example: export LANGCHAIN_TRACING_V2=true"
    ),
}

missing = {k: v for k, v in REQUIRED.items() if not os.getenv(k)}

if missing:
    print("=" * 70)
    print("MISSING REQUIRED ENVIRONMENT VARIABLES")
    print("=" * 70)
    for var, instructions in missing.items():
        print(f"\n  {var}")
        print(f"    {instructions}")
    print("\nSet the missing variables and re-run this cell before continuing.")
    print("=" * 70)
    # Raise so downstream cells don't silently run without credentials.
    raise EnvironmentError(
        f"Missing env vars: {list(missing.keys())}. See instructions above."
    )

# Confirm presence without printing any key bytes.
print("OPENAI_API_KEY       … found ✓")
print("LANGCHAIN_API_KEY    … found ✓")
print("LANGCHAIN_TRACING_V2 … found ✓")

# Optional: show which LangSmith project traces will appear under.
project = os.getenv("LANGCHAIN_PROJECT", "default")
print(f"LangSmith project: '{project}'")
print("Traces will appear at: https://smith.langchain.com")

In [ ]:
from langsmith import traceable, Client
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# Default model — cheap and fast for teaching demos.
MODEL = "gpt-4o-mini"

llm = ChatOpenAI(model=MODEL, temperature=0)
ls_client = Client()

print(f"LLM: {MODEL}")
print("LangSmith Client ready.")

## 2. Trace a Single LLM Call

The simplest trace is one LLM call. Because `LANGCHAIN_TRACING_V2=true` is set, every `ChatOpenAI` invocation is **automatically traced** — no decorator needed. LangSmith captures the prompt messages, the completion, token counts, and latency and sends them to the UI.

We wrap the call in a `@traceable`-decorated function to give the run a named span (`answer_question`) that appears as the **parent** in the trace tree.

After the cell runs, the **run ID** is printed to the cell output. If `run.url` is available on the fetched Run object it will also be printed; in practice `list_runs()`-fetched objects often have `url = None`, so the code prints a fallback instruction instead. Either way, open https://smith.langchain.com, navigate to your project, and open the most recent run to view the trace. Look at:
- **Inputs** tab: the exact messages sent to the model.
- **Outputs** tab: the completion text.
- **Metadata** tab: latency (ms) and token breakdown (prompt / completion / total).

### Try it

> **Gotcha:** The `@traceable` decorator and its `run_type` parameter are part of the `langsmith` SDK and have been stable since `langsmith>=0.1`. If you see `TypeError: traceable() got an unexpected keyword argument`, check that your installed version is recent: `pip install --upgrade langsmith`.

In [ ]:
@traceable(run_type="chain", name="answer_question")
def answer_question(question: str) -> str:
    """Trace a single LLM call. LANGCHAIN_TRACING_V2 auto-traces ChatOpenAI."""
    response = llm.invoke([HumanMessage(content=question)])
    return response.content


# Run it.
answer = answer_question("What is the capital of France, in one word?")
print(f"Answer: {answer}")
print()
# Fetch the most-recent run so we can surface the run ID for the learner.
# We don't build the URL ourselves because the path contains internal IDs;
# instead we direct the learner to the project view.
runs = list(
    ls_client.list_runs(
        project_name=os.getenv("LANGCHAIN_PROJECT", "default"),
        limit=1,
    )
)
if runs:
    run = runs[0]
    print(f"Run ID  : {run.id}")
    # run.url is populated when the Client knows its host URL; may be None.
    if run.url:
        print(f"Trace URL: {run.url}")
    else:
        print("Go to https://smith.langchain.com and open the most recent run in your project.")
else:
    print("Run submitted to LangSmith. Open https://smith.langchain.com to view it.")

## 3. Trace a Multi-Step Agent

A single span is useful but the real power of tracing shows up when the agent has multiple steps. We build a small 3-step agent:

1. **`retrieve_fact(topic)`** — returns a canned fact string (no network call; the interesting part is the span, not the retrieval logic).
2. **`synthesize_answer(topic, fact)`** — calls the LLM to turn the raw fact into a complete sentence.
3. **`format_response(raw_answer)`** — applies a light post-processing transform (uppercases the first letter, ensures a trailing period).

Each step is decorated with `@traceable`, so each becomes its own **child span** nested under the parent `run_agent` span. The LangSmith UI will show:

```
run_agent  (parent span)
├── retrieve_fact   inputs: {topic}    outputs: {fact string}
├── synthesize_answer  inputs: {topic, fact}  outputs: {llm response}
│     └── ChatOpenAI  (auto-traced by LangChain)
└── format_response  inputs: {raw_answer}  outputs: {formatted string}
```

Per span you can read: exact inputs/outputs, wall-clock latency, and (for LLM spans) token counts. This is the view that answers *"what happened inside that agent run?"*

### Try it

In [ ]:
# --- Step 1: retrieve a canned fact (no network; real retrieval would use a
#             vector store or search API — the span structure is identical).
@traceable(run_type="retriever", name="retrieve_fact")
def retrieve_fact(topic: str) -> str:
    """Return a pre-recorded fact about `topic`."""
    facts = {
        "photosynthesis": (
            "Photosynthesis converts carbon dioxide and water into glucose "
            "using light energy, releasing oxygen as a byproduct."
        ),
        "black holes": (
            "A black hole is a region of spacetime where gravity is so strong "
            "that nothing, not even light, can escape once it crosses the "
            "event horizon."
        ),
    }
    return facts.get(topic.lower(), f"No cached fact found for '{topic}'.")


# --- Step 2: ask the LLM to synthesise a clean answer from the raw fact.
@traceable(run_type="llm", name="synthesize_answer")
def synthesize_answer(topic: str, fact: str) -> str:
    """Use the LLM to turn a raw fact into a polished one-sentence explanation."""
    prompt = (
        f"Using only the following fact, write a single clear sentence that "
        f"explains '{topic}' to a curious 12-year-old.\n\nFact: {fact}"
    )
    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()


# --- Step 3: light formatting — ensures sentence casing and a trailing period.
@traceable(run_type="chain", name="format_response")
def format_response(raw_answer: str) -> str:
    """Capitalise the first letter and ensure the sentence ends with a period."""
    answer = raw_answer.strip()
    if answer:
        answer = answer[0].upper() + answer[1:]
    if answer and not answer.endswith((".", "!", "?")):
        answer += "."
    return answer


# --- Parent span that orchestrates all three steps.
@traceable(run_type="chain", name="run_agent")
def run_agent(topic: str) -> str:
    """Run the full 3-step agent and return the formatted answer."""
    fact = retrieve_fact(topic)
    raw_answer = synthesize_answer(topic, fact)
    return format_response(raw_answer)


print("Agent functions defined.")

In [ ]:
result = run_agent("photosynthesis")
print(f"Agent answer: {result}")
print()
print("Open https://smith.langchain.com and find the 'run_agent' run.")
print("You should see 4 spans: run_agent > retrieve_fact, synthesize_answer, format_response.")
print("Click each span to read its inputs, outputs, latency, and token counts.")

## 4. Debug a Broken Run

Tracing earns its keep when something goes wrong. We deliberately introduce a bug: `retrieve_fact_broken` returns a `dict` instead of a `str`. The downstream `synthesize_answer` step receives a dict where it expects a string and the LLM prompt becomes malformed.

Without tracing, debugging this means reading a long stack trace and guessing which step produced the bad value. With tracing, the LangSmith UI shows:

- `run_agent_broken` span: the run completes but produces garbled output.
- `retrieve_fact_broken` span: output is `{"topic": ..., "raw": {...}}` (a dict, not a string).
- `synthesize_answer` span: the input `fact` field is the stringified dict, so the LLM produces nonsense.

The key insight: **the trace pinpoints the span where the bad value was first introduced** (`retrieve_fact_broken`), not just where the failure eventually surfaced. That's the debugging value.

### Try it

> **Gotcha:** When a `@traceable`-decorated function raises an exception, LangSmith marks that span ERROR and still sends the partial trace. The parent span is also marked ERROR. You can always read a failed run's spans — you don't need a successful run to get tracing value.

In [ ]:
@traceable(run_type="retriever", name="retrieve_fact_broken")
def retrieve_fact_broken(topic: str) -> dict:
    """BUG: returns a dict instead of a str. Downstream steps expect a str."""
    # Simulates a retrieval function that accidentally returns raw metadata
    # instead of the fact string.
    return {
        "topic": topic,
        "raw": {
            "content": "Photosynthesis converts CO2 and water into glucose.",
            "source": "biology_db",
            "confidence": 0.97,
        },
    }


@traceable(run_type="chain", name="run_agent_broken")
def run_agent_broken(topic: str) -> str:
    """Run the broken agent. retrieve_fact_broken returns a dict, not a str."""
    fact = retrieve_fact_broken(topic)  # Bug: fact is a dict here.
    # synthesize_answer expects fact: str — passing str(dict) feeds it a Python
    # repr string as context, producing malformed/nonsensical LLM output.
    raw_answer = synthesize_answer(topic, str(fact))  # str(dict) = ugly repr
    return format_response(raw_answer)


print("Running broken agent...")
broken_result = run_agent_broken("photosynthesis")
print(f"Broken agent output: {broken_result!r}")
print()
print("Open https://smith.langchain.com and find the 'run_agent_broken' run.")
print("Inspect the 'retrieve_fact_broken' span — its output is a dict, not a string.")
print("That span is where the bug lives. The LLM received a stringified dict as context,")
print("which explains any garbled output downstream.")

## 5. Latency, Tokens, Cost at a Glance

The LangSmith UI shows per-run latency and token usage visually, but you can also read these programmatically via the `langsmith.Client`. This is useful for building lightweight cost-monitoring scripts or for comparing runs in code (notebook 07 does this at scale inside LangSmith experiments).

The `Client.list_runs()` method returns `Run` objects. The fields most useful for cost analysis are:

| Field | What it contains |
|---|---|
| `run.latency` | Wall-clock latency in seconds (computed property: `end_time - start_time`) |
| `run.total_tokens` | Total tokens consumed by the run (prompt + completion) |
| `run.prompt_tokens` | Tokens in the input messages |
| `run.completion_tokens` | Tokens in the model's response |
| `run.total_cost` | Estimated USD cost as a `Decimal` (populated by LangSmith server-side) |
| `run.status` | `"success"`, `"error"`, etc. |

> **Gotcha:** `total_cost` is populated by LangSmith on the server side using its internal pricing table. If the field is `None`, either the model price is not in LangSmith's table or the SDK version doesn't expose it yet. `latency` and token counts are more reliably populated. The `Run.latency` property (added in recent SDK versions) returns `(end_time - start_time).total_seconds()` — use it directly instead of computing the timedelta manually. Check [LangSmith docs on cost tracking](https://docs.smith.langchain.com/observability/how_to_guides/trace_with_langchain) for current field names.

### Try it

In [ ]:
import time

# Run the working agent once more so we have a fresh run to inspect.
_ = run_agent("black holes")

# Give LangSmith a moment to ingest the run before we query it.
# In practice you'd query asynchronously or poll; here a short wait is fine.
time.sleep(3)

# Fetch recent chain-type runs for this project.
project_name = os.getenv("LANGCHAIN_PROJECT", "default")
recent_runs = list(
    ls_client.list_runs(
        project_name=project_name,
        run_type="chain",
        limit=5,
    )
)

# Find the most recent run_agent run.
agent_run = next(
    (r for r in recent_runs if r.name == "run_agent"),
    None,
)

if agent_run is None:
    print("Could not find a 'run_agent' run in recent results.")
    print("This can happen if the run hasn't been ingested yet. Wait a few seconds and re-run.")
else:
    # Use the Run.latency property (returns float seconds, or None if end_time is absent).
    latency_s = agent_run.latency  # property: (end_time - start_time).total_seconds()
    print(f"Run name     : {agent_run.name}")
    print(f"Run ID       : {agent_run.id}")
    print(f"Status       : {agent_run.status}")
    if latency_s is not None:
        print(f"Latency      : {latency_s:.2f}s")
    else:
        print("Latency      : (not yet available)")
    print(f"Prompt tokens: {agent_run.prompt_tokens}")
    print(f"Completion   : {agent_run.completion_tokens}")
    print(f"Total tokens : {agent_run.total_tokens}")
    if agent_run.total_cost is not None:
        print(f"Total cost   : ${agent_run.total_cost:.6f}")
    else:
        print("Total cost   : (not populated for this model/version)")
    print()
    print("These numbers let you reason about cost per run before moving to")
    print("large-scale experiments (notebook 07 does this across entire datasets).")

## 6. One-Line Note on Alternatives

**Langfuse** (https://langfuse.com) and **Arize Phoenix** (https://phoenix.arize.com) are two well-maintained open-source alternatives that support OpenTelemetry-native tracing and can be self-hosted. This series uses LangSmith because it integrates with the LangChain stack we're already using and has a generous free tier — but the tracing *concepts* (spans, parent/child relationships, inputs/outputs per span) transfer directly to any of these tools.

## What you just learned

- **Tracing** adds a hierarchical call graph to every agent run: each function decorated with `@traceable` becomes a **span** with captured inputs, outputs, latency, and token counts.
- Setting `LANGCHAIN_TRACING_V2=true` enables **automatic tracing** for all LangChain components (like `ChatOpenAI`) without any code changes.
- A **nested span tree** (parent run → child spans → LLM sub-spans) lets you pinpoint exactly which step in a multi-step agent produced a given output — or a given error.
- When a run breaks, the trace **localises the failure**: you can read the output of each span and identify the first span where the data went wrong, rather than hunting through a flat log.
- The `langsmith.Client` lets you fetch run metadata programmatically: `total_tokens`, `prompt_tokens`, `completion_tokens`, `total_cost`, and wall-clock `latency` are all available on the `Run` object.
- LangSmith is a hosted service — traces appear in the UI at https://smith.langchain.com, not inside the notebook.

## What's missing

We can now trace individual runs and inspect them in the UI. But our eval harness from notebook 03 — the `run_eval(agent, dataset, graders)` loop that scores outputs in bulk — still runs entirely offline. Its results live only in a Python list.

**Notebook 07 (`07_datasets_experiments_and_cost.ipynb`)** moves that harness into LangSmith: we'll upload an eval dataset to LangSmith, run a named **experiment** using `Client.evaluate()`, and compare experiments across agent versions in the UI the same way we compared them with diff tables in notebook 04. We'll also see how LangSmith aggregates cost, latency, and token usage across an entire dataset run — giving us the cost dimension we've been missing from our offline evals.